# GSE153164 H5 → H5AD Converter

Convert all RNA and ATAC HDF5 files from GSE153164 into AnnData (.h5ad).


In [1]:
import os, glob, re, traceback
import h5py
import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import csc_matrix

ROOT = '/home/nakagawa/datasets/downloads_but_processed/developing_somatosensory_scRNA_scATAC'
RAW_DIR = os.path.join(ROOT, 'inspect_tar')
OUT_DIR = os.path.join(ROOT, 'h5ad')
os.makedirs(OUT_DIR, exist_ok=True)
print('RAW_DIR =', RAW_DIR)
print('OUT_DIR =', OUT_DIR)


RAW_DIR = /home/nakagawa/datasets/downloads_but_processed/developing_somatosensory_scRNA_scATAC/inspect_tar
OUT_DIR = /home/nakagawa/datasets/downloads_but_processed/developing_somatosensory_scRNA_scATAC/h5ad


In [2]:
def decode(x):
    return x.decode() if isinstance(x, bytes) else str(x)

def parse_stage(fname):
    for p in [r'(E\d+_?\d*)', r'(P\d+)']:
        m = re.search(p, fname)
        if m:
            return m.group(1).replace('_', '.')
    return 'unknown'

def build_csc(data, indices, indptr, shape):
    return csc_matrix((data, indices, indptr), shape=tuple(shape))


In [6]:
def load_rna(fn):

    with h5py.File(fn, "r") as f:

        # ----------------------------
        # Old Cell Ranger v2
        # ----------------------------
        if "mm10" in f:

            grp = f["mm10"]

            X = build_csc(
                grp["data"][:],
                grp["indices"][:],
                grp["indptr"][:],
                grp["shape"][:]
            )

            genes = [decode(x) for x in grp["gene_names"][:]]
            gene_ids = [decode(x) for x in grp["genes"][:]]
            barcodes = [decode(x) for x in grp["barcodes"][:]]

        # ----------------------------
        # New Cell Ranger v3
        # ----------------------------
        elif "matrix" in f:

            grp = f["matrix"]

            X = build_csc(
                grp["data"][:],
                grp["indices"][:],
                grp["indptr"][:],
                grp["shape"][:]
            )

            barcodes = [decode(x) for x in grp["barcodes"][:]]

            feat = grp["features"]

            genes = [decode(x) for x in feat["name"][:]]
            gene_ids = [decode(x) for x in feat["id"][:]]

        else:
            raise RuntimeError(f"Unknown RNA format: {fn}")

    adata = ad.AnnData(X=X.T)

    adata.obs_names = barcodes
    adata.var_names = genes
    adata.var["gene_id"] = gene_ids

    return adata


In [5]:
for fn in sorted(glob.glob(os.path.join(RAW_DIR, "*.h5"))):
    with h5py.File(fn, "r") as f:
        print(os.path.basename(fn), list(f.keys()))

GSM4635072_E11_5_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635073_E12_5_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635074_E13_5_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635075_E14_5_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635076_E15_5_S1_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635077_E16_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635078_E18_5_S1_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635079_E18_S3_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635080_P1_S1_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635081_P1_S2_filtered_gene_bc_matrices_h5.h5 ['mm10']
GSM4635082_Fezf2Het_E15_filtered_feature_bc_matrix.h5 ['matrix']
GSM4635083_Fezf2Het_P1F_filtered_feature_bc_matrix.h5 ['matrix']
GSM4635084_Fezf2Het_P1M_filtered_feature_bc_matrix.h5 ['matrix']
GSM4635085_Fezf2KO_E13_filtered_feature_bc_matrix.h5 ['matrix']
GSM4635086_Fezf2KO_E15_filtered_feature_bc_matrix.h5 ['matrix']
GSM4635087_Fezf2KO_P1_filtered_feature_bc_matrix.h5 ['matrix']
GSM4635088_Fezf2het_E13_filte

In [7]:
# Test one RNA file
rna_file = glob.glob(os.path.join(RAW_DIR, '*feature_bc_matrix*.h5'))[0]
adata = load_rna(rna_file)
print(adata)
adata


AnnData object with n_obs × n_vars = 9788 × 27998
    var: 'gene_id'


AnnData object with n_obs × n_vars = 9788 × 27998
    var: 'gene_id'

In [8]:
# Test one ATAC file
atac_file = glob.glob(os.path.join(RAW_DIR, '*peak_bc_matrix*.h5'))[0]
adata = load_atac(atac_file)
print(adata)
adata


AnnData object with n_obs × n_vars = 12964 × 152179


AnnData object with n_obs × n_vars = 12964 × 152179

In [9]:
# Full conversion
records = []
files = sorted(glob.glob(os.path.join(RAW_DIR, '*.h5')))
print('Found', len(files), 'files')

for fn in files:
    basename = os.path.basename(fn)
    try:
        if 'peak_bc_matrix' in basename:
            modality = 'scATAC'
            adata = load_atac(fn)
        else:
            modality = 'scRNA'
            adata = load_rna(fn)

        adata.obs['sample'] = basename
        adata.obs['modality'] = modality
        adata.obs['stage'] = parse_stage(basename)

        out = os.path.join(OUT_DIR, basename.replace('.h5', '.h5ad'))

        if os.path.exists(out):
            print('SKIP', basename)
            continue

        print('WRITE', basename, adata.shape)
        adata.write_h5ad(out, compression='gzip')

        records.append({'file': basename, 'status': 'success'})
    except Exception:
        traceback.print_exc()
        records.append({'file': basename, 'status': 'failed'})

pd.DataFrame(records).to_csv(os.path.join(OUT_DIR, 'conversion_log.tsv'), sep='\t', index=False)
print('DONE')


Found 23 files
WRITE GSM4635072_E11_5_filtered_gene_bc_matrices_h5.h5 (2757, 27998)
WRITE GSM4635073_E12_5_filtered_gene_bc_matrices_h5.h5 (8015, 27998)
WRITE GSM4635074_E13_5_filtered_gene_bc_matrices_h5.h5 (6415, 27998)
WRITE GSM4635075_E14_5_filtered_gene_bc_matrices_h5.h5 (3485, 27998)
WRITE GSM4635076_E15_5_S1_filtered_gene_bc_matrices_h5.h5 (10891, 27998)
WRITE GSM4635077_E16_filtered_gene_bc_matrices_h5.h5 (5332, 27998)
WRITE GSM4635078_E18_5_S1_filtered_gene_bc_matrices_h5.h5 (6237, 27998)
WRITE GSM4635079_E18_S3_filtered_gene_bc_matrices_h5.h5 (11500, 27998)
WRITE GSM4635080_P1_S1_filtered_gene_bc_matrices_h5.h5 (6552, 27998)
WRITE GSM4635081_P1_S2_filtered_gene_bc_matrices_h5.h5 (4691, 27998)
WRITE GSM4635082_Fezf2Het_E15_filtered_feature_bc_matrix.h5 (10085, 27998)
WRITE GSM4635083_Fezf2Het_P1F_filtered_feature_bc_matrix.h5 (4655, 27998)
WRITE GSM4635084_Fezf2Het_P1M_filtered_feature_bc_matrix.h5 (5288, 27998)
WRITE GSM4635085_Fezf2KO_E13_filtered_feature_bc_matrix.h5 (3669,